## 2. Data Cleaning

### Objective

Clean and standardize the OULAD datasets before feature engineering and modeling.

The cleaning process focuses on:
- Removing exact duplicate records
- Standardizing data types
- Handling missing values according to their meaning
- Preserving information that may be useful for early-risk prediction
- Validating the cleaned datasets before integration

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

RANDOM_STATE = 42

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "raw"

print("Data directory:", DATA_DIR.resolve())

Data directory: D:\Early Risk Detection for Student Success - OULAD\data\raw


In [2]:
student_info = pd.read_csv(DATA_DIR / "studentInfo.csv")
student_registration = pd.read_csv(DATA_DIR / "studentRegistration.csv")
courses = pd.read_csv(DATA_DIR / "courses.csv")
assessments = pd.read_csv(DATA_DIR / "assessments.csv")
student_assessment = pd.read_csv(DATA_DIR / "studentAssessment.csv")
student_vle = pd.read_csv(DATA_DIR / "studentVle.csv")
vle = pd.read_csv(DATA_DIR / "vle.csv")

datasets = {
    "student_info": student_info,
    "student_registration": student_registration,
    "courses": courses,
    "assessments": assessments,
    "student_assessment": student_assessment,
    "student_vle": student_vle,
    "vle": vle
}

print("Datasets loaded successfully.")

Datasets loaded successfully.


In [3]:
raw_datasets = {
    name: df.copy()
    for name, df in datasets.items()
}

## 2.1 Duplicate Removal

Exact duplicate rows were checked across all datasets.

The OULAD tables did not contain meaningful duplicate records except for `studentVle`, where exact duplicate rows were identified.

These duplicates were removed because they represent repeated copies of the same observation and would otherwise overstate student activity.

In [4]:
duplicate_summary = []

for name, df in datasets.items():
    duplicate_summary.append({
        "dataset": name,
        "duplicate_rows": df.duplicated().sum()
    })

pd.DataFrame(duplicate_summary)

,dataset,duplicate_rows
0,student_info,0
1,student_registration,0
2,courses,0
3,assessments,0
4,student_assessment,0
5,student_vle,787170
6,vle,0


In [5]:
before = len(student_vle)

student_vle = student_vle.drop_duplicates().copy()

after = len(student_vle)

print(f"studentVle rows before: {before:,}")
print(f"studentVle rows after:  {after:,}")
print(f"Rows removed:           {before - after:,}")
print(f"Removed %:              {(before - after) / before * 100:.2f}%")

studentVle rows before: 10,655,280
studentVle rows after:  9,868,110
Rows removed:           787,170
Removed %:              7.39%


## Insight

`studentVle` contained 787,170 exact duplicate rows (7.39% of the table).
Removing them prevents repeated activity records from artificially inflating measures of student engagement.

## 2.2 Data Type Standardization

Several OULAD variables are stored as generic `object` or floating-point types even though they represent categorical or integer-like information.

Types are standardized while preserving missing values where they carry meaning.

In [6]:
# Categorical variables
categorical_cols = {
    "student_info": [
        "code_module", "code_presentation", "gender", "region",
        "highest_education", "imd_band", "age_band",
        "disability", "final_result"
    ],
    "student_registration": [
        "code_module", "code_presentation", "activity_type"
    ],
    "courses": [
        "code_module", "code_presentation"
    ],
    "assessments": [
        "code_module", "code_presentation", "assessment_type"
    ],
    "vle": [
        "code_module", "activity_type"
    ]
}

for name, cols in categorical_cols.items():
    for col in cols:
        if col in datasets[name].columns:
            datasets[name][col] = datasets[name][col].astype("category")

In [7]:
integer_cols = {
    "student_info": [
        "id_student", "num_of_prev_attempts", "studied_credits"
    ],
    "student_registration": [
        "id_student"
    ],
    "assessments": [
        "id_assessment"
    ],
    "student_assessment": [
        "id_assessment", "id_student", "is_banked"
    ],
    "student_vle": [
        "id_student", "id_site", "sum_click"
    ],
    "vle": [
        "id_site"
    ]
}

for name, cols in integer_cols.items():
    for col in cols:
        if col in datasets[name].columns:
            datasets[name][col] = datasets[name][col].astype("Int64")

In [8]:
missing_summary = []

for name, df in datasets.items():
    for column, count in df.isna().sum().items():
        if count > 0:
            missing_summary.append({
                "dataset": name,
                "column": column,
                "missing_count": count,
                "missing_pct": round(count / len(df) * 100, 2)
            })

missing_summary = (
    pd.DataFrame(missing_summary)
    .sort_values(["dataset", "missing_pct"], ascending=[True, False])
)

missing_summary

,dataset,column,missing_count,missing_pct
3,assessments,date,11,5.34
4,student_assessment,score,173,0.10
0,student_info,imd_band,1111,3.41
2,student_registration,date_unregistration,22521,69.10
1,student_registration,date_registration,45,0.14
5,vle,week_from,5243,82.39
6,vle,week_to,5243,82.39


In [9]:
student_info["imd_band"].value_counts(dropna=False)

imd_band
20-30%     3654
30-40%     3539
10-20      3516
0-10%      3311
40-50%     3256
50-60%     3124
60-70%     2905
70-80%     2879
80-90%     2762
90-100%    2536
NaN        1111
Name: count, dtype: int64

In [10]:
student_info["imd_band"] = (
    student_info["imd_band"]
    .cat.add_categories("Unknown")
    .fillna("Unknown")
)

## Insight

`imd_band` has missing values for a small proportion of students.
Because the missing value represents unavailable socioeconomic information rather than a measurable numeric value, it is retained as an `Unknown` category instead of being replaced with an arbitrary band.

In [11]:
student_registration["date_unregistration_missing"] = (
    student_registration["date_unregistration"].isna().astype("Int64")
)

In [12]:
student_registration["date_unregistration"].describe()

count    10072.000000
mean        49.757645
std         82.460890
min       -365.000000
25%         -2.000000
50%         27.000000
75%        109.000000
max        444.000000
Name: date_unregistration, dtype: float64

## Insight

Missing `date_unregistration` values are not treated as errors. They indicate that no withdrawal date was recorded for the student-course registration and therefore carry meaningful information about withdrawal status.

In [13]:
print("Missing scores:", student_assessment["score"].isna().sum())

student_assessment.loc[
    student_assessment["score"].isna(),
    "is_score_missing"
] = 1

student_assessment["is_score_missing"] = (
    student_assessment["is_score_missing"]
    .fillna(0)
    .astype("Int64")
)

Missing scores: 173


## Insight

173 assessment records have missing scores. Missing scores are preserved rather than replaced with zero because a missing assessment result does not necessarily mean a score of zero. A missingness indicator is retained for potential modeling use.

In [14]:
assessments["date_missing"] = (
    assessments["date"].isna().astype("Int64")
)

## Insight

11 assessments have missing dates. Eight are unused in `studentAssessment`, while three are linked to 2,865 student-assessment records. These three assessments require special attention because their missing timestamps create temporal uncertainty for early-risk feature construction.

In [15]:
vle[["week_from", "week_to"]].isna().sum()

week_from    5243
week_to      5243
dtype: int64

In [16]:
vle["week_metadata_missing"] = (
    vle[["week_from", "week_to"]]
    .isna()
    .any(axis=1)
    .astype("Int64")
)

## Insight

A large proportion of VLE records have missing `week_from` and `week_to` metadata. These fields are therefore not imputed because their missingness cannot be reliably reconstructed from the available information. The activity date remains the more reliable temporal signal for engagement features.

In [17]:
print(
    "Student IDs not found in studentInfo:",
    (~student_registration["id_student"].isin(
        student_info["id_student"]
    )).sum()
)

print(
    "Assessment IDs not found in assessments:",
    (~student_assessment["id_assessment"].isin(
        assessments["id_assessment"]
    )).sum()
)

print(
    "Site IDs not found in vle:",
    (~student_vle["id_site"].isin(
        vle["id_site"]
    )).sum()
)

Student IDs not found in studentInfo: 0
Assessment IDs not found in assessments: 0
Site IDs not found in vle: 0


## Insight

All foreign-key checks returned zero unmatched IDs, confirming that the cleaned tables remain structurally consistent and can be safely integrated using their defined keys.

In [18]:
final_duplicate_check = {
    name: df.duplicated().sum()
    for name, df in {
        "student_info": student_info,
        "student_registration": student_registration,
        "courses": courses,
        "assessments": assessments,
        "student_assessment": student_assessment,
        "student_vle": student_vle,
        "vle": vle
    }.items()
}

pd.Series(final_duplicate_check)

student_info            0
student_registration    0
courses                 0
assessments             0
student_assessment      0
student_vle             0
vle                     0
dtype: int64

In [19]:
cleaned_datasets = {
    "student_info": student_info,
    "student_registration": student_registration,
    "courses": courses,
    "assessments": assessments,
    "student_assessment": student_assessment,
    "student_vle": student_vle,
    "vle": vle
}

final_missing = []

for name, df in cleaned_datasets.items():
    for col, count in df.isna().sum().items():
        if count > 0:
            final_missing.append({
                "dataset": name,
                "column": col,
                "missing_count": count,
                "missing_pct": round(count / len(df) * 100, 2)
            })

pd.DataFrame(final_missing)

,dataset,column,missing_count,missing_pct
0,student_registration,date_registration,45,0.14
1,student_registration,date_unregistration,22521,69.10
2,assessments,date,11,5.34
3,student_assessment,score,173,0.10
4,vle,week_from,5243,82.39
5,vle,week_to,5243,82.39


In [20]:
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

for name, df in cleaned_datasets.items():
    df.to_csv(PROCESSED_DIR / f"{name}_clean.csv", index=False)

print("Cleaned datasets saved successfully.")

Cleaned datasets saved successfully.


## Data Cleaning Summary

The OULAD datasets were cleaned and validated before feature engineering.

Key actions included:

- Removed 787,170 exact duplicate records from `studentVle`.
- Preserved meaningful missing values such as missing withdrawal dates.
- Represented unavailable socioeconomic information in `imd_band` as `Unknown`.
- Preserved missing assessment scores rather than treating them as zero.
- Added missingness indicators where missingness itself may carry predictive information.
- Standardized categorical and integer-like data types.
- Validated foreign-key relationships across the main OULAD tables.
- Confirmed that no unmatched student, assessment, or VLE site IDs remain.

The cleaned datasets are now ready for feature engineering and early-risk modeling.